# MCS-201 Hardware Acceleration
**Artificial Intelligence and System Engineering · Prince of Songkla University, Phuket Campus**

---

# Lab 3 — Your First CUDA Kernels
### การเขียน คอมไพล์ ดีบัก และวัดผล CUDA kernel ด้วยตนเอง

| | |
|---|---|
| **สัปดาห์** | 4 |
| **วันปฏิบัติการ** | จันทร์ 7 ก.ย. 2569 |
| **กำหนดส่ง** | จันทร์ 7 ก.ย. 2569 **ภายใน 13.00 น.** (วันเดียวกับคาบปฏิบัติการ) |
| **น้ำหนัก** | อยู่ในหมวด Lab ซึ่งคิดเป็น 15% ของคะแนนรายวิชา (ทุกแลปถ่วงน้ำหนักเท่ากัน) |
| **ช่องทางส่ง** | LMS — lms.psu.ac.th |
| **ไฟล์ที่ต้องส่ง** | `Lab3_<รหัสนักศึกษา>.ipynb` พร้อมไฟล์ `.cu` (บีบเป็น .zip) |
| **ผู้สอน** | ผศ.ดร.ฐิตินันท์ เกลี้ยงสุวรรณ · thitinan.kl@phuket.psu.ac.th · ห้อง 6714 |

> **การส่งงาน** อัปโหลดผ่าน LMS ภายใน 13.00 น. ของวันปฏิบัติการ
>
> **การส่งช้า** นับจาก 13.00 น. ของวันปฏิบัติการ หักวันละ 10% ของคะแนนแลปนี้
> และไม่รับหลังพ้นกำหนด 7 วัน (ยกเว้นกรณีมีเหตุจำเป็นและได้รับอนุมัติล่วงหน้า)
>
> **ห้ามล้าง output ก่อนส่ง** ผู้ตรวจต้องเห็นผลลัพธ์ที่รันจริง

> **ทุกคำสั่ง CUDA ต้องผ่าน `CUDA_CHECK`** — คำสั่งที่ไม่ตรวจสอบถูกหักจุดละ 2 คะแนน (สูงสุด 10)

## 1. วัตถุประสงค์ (Objectives)

1. เขียนโปรแกรม CUDA ที่สมบูรณ์ตั้งแต่การจองหน่วยความจำจนถึงการคืนหน่วยความจำ — *CLO2*
2. ใช้การตรวจสอบข้อผิดพลาดอย่างเป็นระบบด้วยมาโคร `CUDA_CHECK` — *CLO2*
3. เข้าใจผลของการลืมตรวจสอบขอบเขตและการลืมซิงโครไนซ์ ผ่านการทดลองทำให้พังโดยตั้งใจ — *CLO2, CLO3*
4. วัดแบนด์วิดท์และ TFLOPS ที่ทำได้จริงของ kernel ที่เขียนเอง แล้วเทียบกับเพดานจาก Lab 2 — *CLO3*

## 2. การคอมไพล์บน Google Colab

ใช้ `%%writefile` เขียนไฟล์ `.cu` แล้วเรียก `nvcc` ผ่าน shell

In [16]:
!nvcc --version
!nvidia-smi --query-gpu=name,compute_cap --format=csv

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
name, compute_cap
Tesla T4, 7.5


In [17]:
ARCH = "sm_75"      # Tesla T4 (compute capability 7.5)
                    # sm_75 = T4 · sm_80 = A100 · sm_89 = L4 · sm_90 = H100
print("using", ARCH)

using sm_75


In [18]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("block %d, thread %d\n", blockIdx.x, threadIdx.x);
}

int main() {
    hello<<<2, 4>>>();        /* 2 blocks x 4 threads */
    cudaDeviceSynchronize();  /* without this, main() may */
    return 0;                 /* exit before the GPU prints */
}

Overwriting hello.cu


In [19]:
!nvcc -arch={ARCH} hello.cu -o hello && ./hello

block 1, thread 0
block 1, thread 1
block 1, thread 2
block 1, thread 3
block 0, thread 0
block 0, thread 1
block 0, thread 2
block 0, thread 3


---
## 3. ขั้นตอนการปฏิบัติ

### ส่วนที่ 1 — Vector addition ที่สมบูรณ์

เติมส่วน `TODO` ทั้ง 4 จุดให้ครบ

In [20]:
%%writefile vecadd.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call)                                        \
  do {                                                          \
    cudaError_t err = (call);                                   \
    if (err != cudaSuccess) {                                   \
      fprintf(stderr, "CUDA %s:%d: %s\n", __FILE__, __LINE__,   \
              cudaGetErrorString(err));                         \
      exit(EXIT_FAILURE);                                       \
    }                                                           \
  } while (0)

__global__ void vecAdd(int n, const float *x, float *y) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = x[i] + y[i];
}

int main() {
    const int n = 1 << 24;                     // 16.7 M
    const size_t bytes = n * sizeof(float);

    float *h_x = (float*)malloc(bytes);
    float *h_y = (float*)malloc(bytes);
    for (int i = 0; i < n; i++) { h_x[i] = 1.0f; h_y[i] = 2.0f; }

    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, bytes));
    CUDA_CHECK(cudaMalloc(&d_y, bytes));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, h_y, bytes, cudaMemcpyHostToDevice));

    int threads = 256;
    int blocks  = (n + threads - 1) / threads;   // ปัดเศษขึ้นเสมอ

    cudaEvent_t start, stop; float ms = 0.0f;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    vecAdd<<<blocks, threads>>>(n, d_x, d_y);    // warm-up
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaEventRecord(start));
    for (int r = 0; r < 20; r++) {
        vecAdd<<<blocks, threads>>>(n, d_x, d_y);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
    ms /= 20.0f;

    CUDA_CHECK(cudaMemcpy(h_y, d_y, bytes, cudaMemcpyDeviceToHost));
    const float expected = 23.0f;  // เริ่มที่ 2 และบวก x=1 จำนวน 21 รอบ
    int errors = 0;
    for (int i = 0; i < n; i++) {
        if (fabsf(h_y[i] - expected) > 1e-5f) {
            if (errors < 5)
                fprintf(stderr, "mismatch at %d: got %.1f expected %.1f\n",
                        i, h_y[i], expected);
            errors++;
        }
    }
    printf("verification: %s (expected y[i] = %.1f)\n",
           errors == 0 ? "PASS" : "FAIL", expected);

    double gbps = 12.0 * n / (ms * 1e-3) / 1e9;
    printf("vecAdd : %8.3f ms   %7.1f GB/s   %5.1f%% of Lab 2\n",
           ms, gbps, gbps / 241.0 * 100.0);

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));
    free(h_x);
    free(h_y);
    return 0;
}

Overwriting vecadd.cu


In [21]:
!nvcc -O3 -arch={ARCH} vecadd.cu -o vecadd && ./vecadd

verification: PASS (expected y[i] = 23.0)
vecAdd :    0.767 ms     262.3 GB/s   108.8% of Lab 2


---
### ส่วนที่ 2 — ทำให้พังโดยตั้งใจ

สร้างสำเนาที่ **ลบการตรวจสอบขอบเขตออก** แล้วรันด้วย `compute-sanitizer`

In [22]:
%%writefile vecadd_bug.cu
/* สำเนาที่ (ก) ตัดการตรวจสอบขอบเขตออก และ (ข) ใช้ n ที่หารด้วย 256 ไม่ลงตัว */
#include <cstdio>
#include <cstdlib>

__global__ void vecAdd(int n, const float *x, float *y) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    y[i] = x[i] + y[i];              /* ไม่มี if (i < n) */
}

int main() {
    const int n = 1000000;           /* 1,000,000 / 256 = 3906.25 -> ไม่ลงตัว */
    const size_t bytes = n * sizeof(float);

    float *d_x, *d_y;
    cudaMalloc(&d_x, bytes);
    cudaMalloc(&d_y, bytes);
    cudaMemset(d_x, 0, bytes);
    cudaMemset(d_y, 0, bytes);

    int threads = 256;
    int blocks  = (n + threads - 1) / threads;      /* = 3907 blocks */
    printf("threads ทั้งหมด = %d, n = %d, เธรดส่วนเกิน = %d\n",
           blocks * threads, n, blocks * threads - n);

    vecAdd<<<blocks, threads>>>(n, d_x, d_y);
    cudaDeviceSynchronize();

    cudaFree(d_x); cudaFree(d_y);
    return 0;
}

Overwriting vecadd_bug.cu


In [23]:
!nvcc -O3 -arch={ARCH} vecadd_bug.cu -o vecadd_bug
!which compute-sanitizer || ls /usr/local/cuda/bin/compute-sanitizer
!compute-sanitizer --tool memcheck ./vecadd_bug 2>&1 | head -30

/usr/local/cuda/bin/compute-sanitizer
========= COMPUTE-SANITIZER
========= Invalid __global__ read of size 4 bytes
=========     at vecAdd(int, const float *, float *)+0x70
=========     by thread (64,0,0) in block (3906,0,0)
=========     Address 0x7e1e9f3d0900 is out of bounds
=========     and is 1 bytes after the nearest allocation at 0x7e1e9f000000 of size 4,000,000 bytes
=========     Saved host backtrace up to driver entry point at kernel launch time
=========         Host Frame: main [0x8a6b] in vecadd_bug
========= Invalid __global__ read of size 4 bytes
=========     at vecAdd(int, const float *, float *)+0x70
=========     by thread (65,0,0) in block (3906,0,0)
=========     Address 0x7e1e9f3d0904 is out of bounds
=========     and is 5 bytes after the nearest allocation at 0x7e1e9f000000 of size 4,000,000 bytes
=========     Saved host backtrace up to driver entry point at kernel launch time
=========         Host Frame: main [0x8a6b] in vecadd_bug
========= Invalid __glob

**บันทึกข้อความที่ `compute-sanitizer` รายงาน และอธิบายว่าเกิดอะไรขึ้น**

ในคำอธิบายให้ตอบด้วยว่า ถ้าเปลี่ยน `n` กลับเป็น 2²⁴ (16,777,216) แล้วรัน sanitizer ใหม่
จะยังพบข้อผิดพลาดหรือไม่ เพราะเหตุใด

_ตอบ:_ `compute-sanitizer` รายงาน `Invalid __global__ read/write of size 4 bytes` เพราะมีทั้งหมด 1,000,192 เธรด แต่ข้อมูลมี 1,000,000 ตัว จึงเกินมา 192 เธรดและเข้าถึงหน่วยความจำนอกขอบเขต ถ้าใช้ n = 2²⁴ จะไม่เกิด error นี้ เพราะ n หารด้วย 256 ลงตัวและไม่มีเธรดส่วนเกิน

**เปรียบเทียบการจับเวลาโดยมี/ไม่มีการซิงโครไนซ์**

รันเซลล์ถัดไป ซึ่งจับเวลา kernel เดียวกันสามวิธี แล้วบันทึกตัวเลขทั้งสามค่า

In [24]:
%%writefile timing.cu
#include <cstdio>
#include <chrono>

__global__ void vecAdd(int n, const float *x, float *y) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = x[i] + y[i];
}

int main() {
    const int n = 1 << 24;
    const size_t bytes = n * sizeof(float);
    float *d_x, *d_y;
    cudaMalloc(&d_x, bytes); cudaMalloc(&d_y, bytes);
    cudaMemset(d_x, 0, bytes); cudaMemset(d_y, 0, bytes);

    int threads = 256, blocks = (n + threads - 1) / threads;
    vecAdd<<<blocks, threads>>>(n, d_x, d_y);      /* warm-up */
    cudaDeviceSynchronize();

    /* (1) chrono โดยไม่ซิงโครไนซ์ -- ผิด */
    auto t0 = std::chrono::steady_clock::now();
    for (int r = 0; r < 20; r++) vecAdd<<<blocks, threads>>>(n, d_x, d_y);
    auto t1 = std::chrono::steady_clock::now();
    double ms_wrong =
        std::chrono::duration<double, std::milli>(t1 - t0).count() / 20;

    /* (2) chrono พร้อมซิงโครไนซ์ -- ถูก */
    cudaDeviceSynchronize();
    t0 = std::chrono::steady_clock::now();
    for (int r = 0; r < 20; r++) vecAdd<<<blocks, threads>>>(n, d_x, d_y);
    cudaDeviceSynchronize();
    t1 = std::chrono::steady_clock::now();
    double ms_right =
        std::chrono::duration<double, std::milli>(t1 - t0).count() / 20;

    /* (3) CUDA events -- วิธีมาตรฐาน */
    cudaEvent_t s, e; float ms_ev = 0.0f;
    cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    for (int r = 0; r < 20; r++) vecAdd<<<blocks, threads>>>(n, d_x, d_y);
    cudaEventRecord(e); cudaEventSynchronize(e);
    cudaEventElapsedTime(&ms_ev, s, e); ms_ev /= 20.0f;

    printf("chrono ไม่ sync : %10.4f ms\n", ms_wrong);
    printf("chrono + sync   : %10.4f ms\n", ms_right);
    printf("CUDA events     : %10.4f ms\n", ms_ev);
    printf("อัตราส่วนผิดพลาด : %10.1f เท่า\n", ms_right / ms_wrong);

    cudaFree(d_x); cudaFree(d_y);
    return 0;
}

Overwriting timing.cu


In [25]:
!nvcc -O3 -arch={ARCH} timing.cu -o timing && ./timing

chrono ไม่ sync :     0.0028 ms
chrono + sync   :     0.7681 ms
CUDA events     :     0.7678 ms
อัตราส่วนผิดพลาด :      271.0 เท่า


**อธิบายผลที่ได้ (3–5 ประโยค)**

_ตอบ:_ แบบไม่ sync ได้เวลาน้อยผิดจริง เพราะ CPU วัดแค่เวลาสั่งงาน kernel โดยไม่ได้รอ GPU ทำเสร็จ เมื่อใส่ `cudaDeviceSynchronize()` เวลาจะใกล้กับ CUDA events มากขึ้น CUDA events จึงเหมาะสำหรับวัดเวลา kernel มากกว่า

---
### ส่วนที่ 3 — SAXPY

เขียน kernel `saxpy`: `y[i] = a*x[i] + y[i]` แล้ววัดแบนด์วิดท์ที่ทำได้

In [26]:
%%writefile saxpy.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call)                                        \
  do {                                                          \
    cudaError_t err = (call);                                   \
    if (err != cudaSuccess) {                                   \
      fprintf(stderr, "CUDA %s:%d: %s\n", __FILE__, __LINE__,   \
              cudaGetErrorString(err));                         \
      exit(EXIT_FAILURE);                                       \
    }                                                           \
  } while (0)

__global__ void saxpy(int n, float a, const float *x, float *y) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = a * x[i] + y[i];
}

int main() {
    const int n = 1 << 24;
    const size_t bytes = (size_t)n * sizeof(float);
    const float a = 2.0f;

    float *h_x = (float*)malloc(bytes);
    float *h_y = (float*)malloc(bytes);
    if (!h_x || !h_y) { fprintf(stderr, "host allocation failed\n"); return 1; }
    for (int i = 0; i < n; i++) { h_x[i] = 1.0f; h_y[i] = 2.0f; }

    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, bytes));
    CUDA_CHECK(cudaMalloc(&d_y, bytes));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, h_y, bytes, cudaMemcpyHostToDevice));

    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    cudaEvent_t start, stop;
    float ms = 0.0f;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    saxpy<<<blocks, threads>>>(n, a, d_x, d_y);  // warm-up
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaEventRecord(start));
    for (int r = 0; r < 20; r++) {
        saxpy<<<blocks, threads>>>(n, a, d_x, d_y);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
    ms /= 20.0f;

    CUDA_CHECK(cudaMemcpy(h_y, d_y, bytes, cudaMemcpyDeviceToHost));
    const float expected = 44.0f;  // 2 + 21*(2*1)
    int errors = 0;
    for (int i = 0; i < n; i++)
        if (fabsf(h_y[i] - expected) > 1e-5f) errors++;

    const double gbps = 12.0 * n / (ms * 1e-3) / 1e9;
    printf("verification: %s (expected y[i] = %.1f)\n",
           errors == 0 ? "PASS" : "FAIL", expected);
    printf("saxpy : %8.3f ms   %7.1f GB/s   %5.1f%% of Lab 2\n",
           ms, gbps, gbps / 241.0 * 100.0);

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));
    free(h_x);
    free(h_y);
    return 0;
}

Overwriting saxpy.cu


In [27]:
!nvcc -O3 -arch={ARCH} saxpy.cu -o saxpy && ./saxpy

verification: PASS (expected y[i] = 44.0)
saxpy :    0.767 ms     262.3 GB/s   108.9% of Lab 2


---
### ส่วนที่ 4 — Naive matrix multiplication

In [28]:
%%writefile matmul.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call)                                        \
  do {                                                          \
    cudaError_t err = (call);                                   \
    if (err != cudaSuccess) {                                   \
      fprintf(stderr, "CUDA %s:%d: %s\n", __FILE__, __LINE__,   \
              cudaGetErrorString(err));                         \
      exit(EXIT_FAILURE);                                       \
    }                                                           \
  } while (0)

__global__ void matmul(int N, const float *A, const float *B, float *C) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
            sum += A[row * N + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

int main(int argc, char **argv) {
    int N = (argc > 1) ? atoi(argv[1]) : 1024;
    const size_t count = (size_t)N * N;
    const size_t bytes = count * sizeof(float);
    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);
    if (!h_A || !h_B || !h_C) { fprintf(stderr, "host allocation failed\n"); return 1; }
    for (size_t i = 0; i < count; i++) { h_A[i] = 1.0f; h_B[i] = 1.0f; }

    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));
    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));

    dim3 threads(16, 16);
    dim3 blocks((N + 15) / 16, (N + 15) / 16);
    matmul<<<blocks, threads>>>(N, d_A, d_B, d_C);  // warm-up
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    const int repeats = 3;
    cudaEvent_t start, stop;
    float ms = 0.0f;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));
    for (int r = 0; r < repeats; r++) {
        matmul<<<blocks, threads>>>(N, d_A, d_B, d_C);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
    ms /= repeats;

    CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost));
    int errors = 0;
    for (size_t i = 0; i < count; i++)
        if (fabsf(h_C[i] - (float)N) > 1e-3f) errors++;
    const double tflops = 2.0 * N * (double)N * N / (ms * 1e-3) / 1e12;
    printf("N=%d : %8.3f ms   %7.4f TFLOPS   %5.1f%% of Lab 2   %s\n",
           N, ms, tflops, tflops / 8.14 * 100.0,
           errors == 0 ? "PASS" : "FAIL");

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaFree(d_C));
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Overwriting matmul.cu


In [29]:
!nvcc -O3 -arch={ARCH} matmul.cu -o matmul
!./matmul 512
!./matmul 1024
!./matmul 2048

N=512 :    1.094 ms    0.2455 TFLOPS     3.0% of Lab 2   PASS
N=1024 :    4.950 ms    0.4338 TFLOPS     5.3% of Lab 2   PASS
N=2048 :   37.815 ms    0.4543 TFLOPS     5.6% of Lab 2   PASS


### ตารางที่ 1 — สรุปผลการวัด

| Kernel | ขนาด | เวลา (ms) | GB/s หรือ TFLOPS | % ของเพดาน | เพดานที่ใช้เทียบ |
|---|---|---|---|---|---|
| vecAdd | n = 2²⁴ | 0.767 | 262.3 GB/s | 108.8% | 241 GB/s จาก Lab 2 |
| saxpy | n = 2²⁴ | 0.767 | 262.3 GB/s | 108.9% | 241 GB/s จาก Lab 2 |
| matmul | N = 512 | 1.094 | 0.2455 TFLOPS | 3.0% | 8.14 TFLOPS จาก Lab 2 |
| matmul | N = 1024 | 4.950 | 0.4338 TFLOPS | 5.3% | 8.14 TFLOPS จาก Lab 2 |
| matmul | N = 2048 | 37.815 | 0.4543 TFLOPS | 5.6% | 8.14 TFLOPS จาก Lab 2 |

> **เก็บตัวเลขชุดนี้ไว้** สัปดาห์ที่ 5 จะนำ `matmul` ตัวเดียวกันนี้มาปรับด้วย shared memory tiling
> แล้ววัด speedup เทียบกับ baseline ชุดนี้

---
## 4. คำถามวิเคราะห์

**Q1.** `vecAdd` และ `saxpy` ทำได้กี่ % ของแบนด์วิดท์สูงสุด ยังมีช่องปรับปรุงหรือไม่ เพราะเหตุใด

_ตอบ:_ vecAdd ทำได้ 108.8% และ saxpy ทำได้ 108.9% ของ bandwidth จาก Lab 2 ทั้งสอง kernel ถูกจำกัดด้วย bandwidth จึงปรับปรุงได้ไม่มาก

**Q2.** `matmul` แบบ naive ทำได้เพียงไม่กี่ % ของ peak — สมาชิกแต่ละตัวของ A ถูกอ่านจาก global memory กี่ครั้ง

_ตอบ:_ สมาชิกแต่ละตัวของ A ถูกอ่าน N ครั้ง เพราะถูกใช้คำนวณผลลัพธ์ทุกคอลัมน์ในแถวนั้น

**Q3.** การอ่าน `B[k*N + col]` เป็น coalesced หรือไม่ แล้ว `A[row*N + k]` ล่ะ
อธิบายโดยอ้างอิงว่าเธรดที่ติดกันใน warp เข้าถึงที่อยู่ใด

_ตอบ:_ การอ่าน B เป็น coalesced เพราะเธรดที่อยู่ติดกันอ่านตำแหน่ง col ที่ต่อกัน ส่วน A เธรดในแถวเดียวกันอ่านตำแหน่งเดียวกัน จึงเป็นการใช้ค่าเดิมซ้ำ ไม่ใช่การอ่านตำแหน่งต่อกันแบบ B

**Q4.** เมื่อเพิ่ม N จาก 512 เป็น 2048 ค่า TFLOPS เปลี่ยนแปลงอย่างไร เพราะเหตุใด

_ตอบ:_ เมื่อเพิ่ม N จาก 512 เป็น 2048 ประสิทธิภาพเพิ่มจาก 0.2455 เป็น 0.4543 TFLOPS เพราะ GPU มีงานมากขึ้น แต่ยังต่ำกว่า peak เนื่องจากอ่าน global memory ซ้ำและไม่ได้ใช้ shared memory

---
## 5. การเตรียมไฟล์ส่งงาน

รันเซลล์ถัดไปเพื่อบีบไฟล์ `.cu` ทั้งหมดแล้วดาวน์โหลด จากนั้นอัปโหลดพร้อมโน้ตบุ๊กนี้ผ่าน LMS

> **ก่อนรัน** แก้ `sid` ในเซลล์ถัดไปให้เป็นรหัสนักศึกษาของท่าน
> ชื่อไฟล์ที่ผิดจะทำให้ระบบตรวจงานหาไฟล์ไม่เจอ

In [30]:
import os
sid = "6730614023"
!zip -q Lab3_{sid}.zip *.cu
print("ไฟล์ที่บีบแล้ว:", os.path.getsize(f"Lab3_{sid}.zip"), "bytes")
try:
    from google.colab import files
    files.download(f"Lab3_{sid}.zip")
except Exception as e:
    print("ดาวน์โหลดเองจากแถบไฟล์ด้านซ้าย:", e)

ไฟล์ที่บีบแล้ว: 5594 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 6. รายการสิ่งที่ต้องส่ง

- [ ] โน้ตบุ๊ก `Lab3_<รหัสนักศึกษา>.ipynb` รันครบทุกเซลล์พร้อม output
- [ ] ไฟล์ `vecadd.cu`, `saxpy.cu`, `matmul.cu` (บีบรวมเป็น .zip พร้อมโน้ตบุ๊ก)
- [ ] ทุกคำสั่ง CUDA มีการตรวจสอบข้อผิดพลาด
- [ ] ผลการตรวจสอบความถูกต้องเทียบกับค่าที่คำนวณด้วยมือ ของทั้งสาม kernel
- [ ] ข้อความจาก `compute-sanitizer` ในส่วนที่ 2
- [ ] ตารางสรุปผลการวัด (ตารางที่ 1) ครบทุกช่อง
- [ ] คำตอบคำถามวิเคราะห์ Q1–Q4

## 7. เกณฑ์การให้คะแนน

| องค์ประกอบ | คะแนน | เกณฑ์การพิจารณา |
|---|---|---|
| ความถูกต้องของ kernel | 40 | ทั้งสาม kernel ให้ผลตรงกับค่าที่คำนวณได้ และมีการตรวจสอบขอบเขต |
| การตรวจสอบข้อผิดพลาด | 20 | ใช้ `CUDA_CHECK` ครบทุกคำสั่ง และตรวจสองขั้นหลังเรียก kernel |
| การวัดผล | 20 | ใช้ CUDA events ถูกต้อง คำนวณแบนด์วิดท์/TFLOPS ถูกหน่วย |
| คำตอบคำถามวิเคราะห์ | 20 | Q1–Q4 ข้อละ 5 โดยเฉพาะ Q2 และ Q3 ต้องอ้างอิงหลักการ |
| **รวม** | **100** | |

**หักคะแนนเพิ่มเติม** คำสั่ง CUDA ที่ไม่ตรวจสอบข้อผิดพลาด หักจุดละ 2 คะแนน (สูงสุด 10 คะแนน)

> **หมายเหตุเรื่องคะแนน** ตาราง 100 คะแนนด้านบนเป็นคะแนนภายในของแลปนี้
> ก่อนแปลงเป็นสัดส่วนในหมวด Lab (15% ของคะแนนรายวิชา)